In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import LeagueOpponentController, LeagueHaxballEnv
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import FixedMultiAgentReset
from src.rl.reward_shapers import Stage3SelfPlayReward
from src.rl.trainer import train_ppo_league

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

POOL_DIR = "models/stage3_league/pool"

def make_league_env():
    def _init():
        roster = [
            PlayerSlot(
                team="red", stats=PlayerStats(name="RL_Agent", accel=3200.0)
            ),
            PlayerSlot(
                team="blue",
                stats=PlayerStats(name="League_Bot", accel=3200.0),
                controller=LeagueOpponentController(pool_dir=POOL_DIR, device="cpu"),
            ),
        ]

        match_cfg = MatchConfig(
            mode=ClassicMatchMode(time_limit=10.0, score_limit=1),
            roster=roster,
            time_limit=10.0,
            score_limit=1,
        )

        return LeagueHaxballEnv(
            match_config=match_cfg,
            reward_shaper=Stage3SelfPlayReward(),
            reset_strategy=FixedMultiAgentReset(offset_x=180.0),
            max_steps=600,
        )
    return _init


NUM_ENVS = 16
OBS_DIM = 80

train_envs = gym.vector.AsyncVectorEnv([make_league_env() for _ in range(NUM_ENVS)])
eval_env = make_league_env()()

model = ActorCritic(obs_dim=OBS_DIM).to(device)

# Load the best Phase D model as the starting point
phase_d_ckpt = "models/stage2_scratch/phaseD/best_model.pt"
model.load_state_dict(torch.load(phase_d_ckpt, map_location=device, weights_only=False))
print(f"✅ Bootstrapping League from Phase D Checkpoint")

train_ppo_league(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=15_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=30,
    save_dir="models/stage3_league",
    pool_dir=POOL_DIR,
    lr_initial=5e-5,
    lr_final=5e-6,
    ent_coef_initial=0.005,
    ent_coef_final=0.0002,
)

train_envs.close()

⚡ Device: cuda
✅ Loaded active self-play learner from models/stage2_scratch/phaseD/best_model.pt
✅ Loaded frozen Stage 2 benchmark model
🚀 Multi-Agent Self-Play | Batch Size: 8192

📊 [EVALUATION @ Step  106496] vs Stage1: Win  0.0% | Loss  0.0% | Draw 100.0% || vs Heuristic: Win  0.0% | Loss 100.0% | Draw  0.0%
   ⭐ New best self-play model saved: models/stage3/best_selfplay_model.pt


KeyboardInterrupt: 

In [4]:
torch.save(
        model.state_dict(), os.path.join("models/stage3", "self_play.pt")
    )

# Test

In [6]:
import torch
from src.rl.ppo_core import ActorCritic
from src.rl.benchmarker import RLController, run_arena, run_solo_drill, render_match
from config.match_config import PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController

device = torch.device("cpu") # Fast inference on CPU

# 1. Load RL models
obs_dim = 80
stage2_model = ActorCritic(obs_dim).to(device)
stage2_model.load_state_dict(torch.load("models/stage3/self_play.pt", map_location=device))


# 2. Setup Team Coordinators
red_rl_controller = RLController(stage2_model, team="red", device=device)

red_heuristic_coord = TeamHeuristicCoordinator(team="red")
red_heuristic_controller = HeuristicBotController(red_heuristic_coord)

blue_heuristic_coord = TeamHeuristicCoordinator(team="blue")
blue_heuristic_controller = HeuristicBotController(blue_heuristic_coord)

blue_rl_controller = RLController(stage2_model, team="blue")



In [7]:
# ==========================================
# TEST 1: The Diagnostic Solo Drill
# ==========================================
print("--- TEST 1: EMPTY NET DIAGNOSTIC ---")
run_solo_drill(
    agent_roster=[PlayerSlot("red", PlayerStats("RL_Test"), red_rl_controller)],
    num_episodes=5,
    time_limit=60.0
)


--- TEST 1: EMPTY NET DIAGNOSTIC ---
🎯 Running Solo Drill: 60.0s per episode (5 Episodes)
   Episode 1: 0 goals
   Episode 2: 0 goals
   Episode 3: 0 goals
   Episode 4: 0 goals
   Episode 5: 0 goals
📊 Average Scoring Rate: 0.00 goals / 60.0s



0.0

In [8]:
# ==========================================
# TEST 2: The Arena 
# RL Agent (Red) vs Heuristic Bot (Blue)
# ==========================================
print("\n--- TEST 2: THE ARENA (1v1) ---")
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]

stats = run_arena(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=10,
    time_limit=60.0,  # 60 second matches
    score_limit=3
)




--- TEST 2: THE ARENA (1v1) ---
🏟️ Running Arena: 1 RED vs 1 BLUE (10 Matches)
✅ Completed in 65.35s
🏆 Series Outcome (Wins): RED 0 | BLUE 8 | DRAWS 2
⚽ Avg Goals / Match:     RED 0.70 | BLUE 2.20



In [9]:
# ==========================================
# TEST 3: Kaggle-Style Visualization
# Watch the matchup in HTML format
# ==========================================
red_roster = [PlayerSlot("red", PlayerStats("RL_Agent"), red_rl_controller)]
blue_roster = [PlayerSlot("blue", PlayerStats("Bot"), blue_heuristic_controller)]


print("\n--- TEST 3: RENDER MATCH ---")
render_match(
    red_roster=red_roster,
    blue_roster=blue_roster,
    num_matches=1,
    time_limit=90.0,
    save_path="renders/arena"
)



--- TEST 3: RENDER MATCH ---
🎬 Generating 1 replays...
Game 1 Result: BLUE WINS! 🎉 (2 - 1)
Replay saved to: renders/arena/2026-08-23_07-35-09_match_1.html



In [11]:
from src.rl.benchmarker import render_solo_drill

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
agent_slot = PlayerSlot("red", PlayerStats("RL_Agent"), RLController(model, team="red", device=device))

# Render 3 randomized episodes (20 seconds each)
replay_path = render_solo_drill(
    agent_slot=agent_slot,
    num_episodes=3,
    time_limit=20.0,
    save_path="renders/solo_drills",
)

🎬 Generating 3 Solo Drill Replays (20.0s each)...
   Episode 1 Finished: 0 Goals Scored
   Episode 2 Finished: 0 Goals Scored
   Episode 3 Finished: 0 Goals Scored
🏆 Overall: 0.00 Avg Goals / 20.0s
Replay saved to: renders/solo_drills/2026-08-23_07-35-33_solo_drill.html

